In [1]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel

# BioBERT model setup
model_name = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Define terms to check
columns_to_check = ['Genome Location', 'Tier', 'Hallmark', 'Chr Band', 'Somatic',
                    'Germline', 'Tissue Type', 'Molecular Genetics', 'Mutation Types',
                    'Other Germline Mut']

def calculate_similarity(embedding1, embedding2):
    return torch.nn.functional.cosine_similarity(embedding1, embedding2, dim=-1).item()

def check_biobert_terms_json(file_path, columns_to_check, output_path, threshold=0.5):
    try:
        # Load the JSON file
        with open(file_path, 'r') as file:
            data = json.load(file)
        print("File loaded successfully!")
    except Exception as e:
        print(f"Error loading the file: {e}")
        return

    # Check if 'biology_paragraphs' key exists
    if 'biology_paragraphs' not in data:
        print("'biology_paragraphs' key not found in the file.")
        return

    # Extract biology paragraphs
    biology_paragraphs = data['biology_paragraphs']
    if not isinstance(biology_paragraphs, list):
        print("Error: 'biology_paragraphs' must be a list of paragraphs.")
        return

    # Precompute term embeddings
    term_embeddings = {}
    for term in columns_to_check:
        tokens = tokenizer(term, return_tensors='pt')
        with torch.no_grad():
            term_embeddings[term] = model(**tokens).last_hidden_state.mean(dim=1)

    # Initialize a DataFrame to store results
    results = pd.DataFrame(columns=['Paragraph'] + columns_to_check)

    # Process each paragraph
    for idx, paragraph in enumerate(biology_paragraphs):
        # Truncate paragraph to 512 tokens
        paragraph_tokens = tokenizer(paragraph, truncation=True, padding=True, max_length=512, return_tensors='pt')
        with torch.no_grad():
            paragraph_embedding = model(**paragraph_tokens).last_hidden_state.mean(dim=1)

        # Check similarity for each term
        row = {'Paragraph': paragraph[:100] + '...'}  # Store the first 100 characters of the paragraph
        for term, term_embedding in term_embeddings.items():
            similarity = calculate_similarity(paragraph_embedding, term_embedding)
            row[term] = 1 if similarity >= threshold else 0  # Store 1 if above threshold, else 0

        # Convert row to DataFrame and append using pd.concat
        row_df = pd.DataFrame([row])
        results = pd.concat([results, row_df], ignore_index=True)

    # Save results to an Excel file
    results.to_excel(output_path, index=False)
    print(f"Results saved to: {output_path}")

# Provide file path
file_path = input("Enter the path to the JSON file you want to check: ")
output_path = "/content/output_results_1.xlsx"  # Output`` Excel file
check_biobert_terms_json(file_path, columns_to_check, output_path)


ModuleNotFoundError: No module named 'transformers'